**NOTE** `phimats` environment should be used as kernel

In [2]:
import numpy as np
import h5py

from PreProcessing import PhysicsConfig, MeshConfig, PreProcessing as PP

from MeshManager import MeshManager

from BoundaryConditions import *

from PostProcessing import WriteXDMF

%load_ext autoreload
%autoreload 2

### Simulation data

In [3]:
SimulName = "300_eq_f4_k1_dis"
RVEName = "RVE"

# Element sets
nElementSets = 1
# Simulation type
SimulType = "2PhaseTrapping" 
# Number of steps to achieve the load
nSteps = 100

### Read mesh file 

In [4]:
# Element name
elementName = "triangle"  		# meshio compatible element name
mesh = MeshManager("RVE_10um.msh", elementName)
mesh.WriteMesh("RVE")

Reading mesh: RVE_10um.msh...

Mesh consistency verified: All domain elements are <triangle>.

PHIMATS MESH INITIATION
Detected 1 Material Sets. Follow this order in your Input Data:
  [Index 1] -> Physical Group: 'RVE'

------------------------------
MESH SUMMARY
------------------------------
File Name:      RVE_10um.msh
Element Type:   triangle
Dimension:      2D
Node Order:     1
Total Nodes:    288106
Total Elements: 574912
Cell Blocks:
  - line: 1298 items
  - triangle: 574912 items
------------------------------



In [5]:
# Create the config object
meshConfig = MeshConfig(
    nTotNodes=mesh.get_nTotNodes(),
    nTotElements=mesh.get_nTotElements(),
    nDim=mesh.get_nDim(),
    materialNames=mesh.getMaterialNames(),
)

### Read phase-field RVE data

In [6]:
# gPhi mapped to finite element mesh

with h5py.File(RVEName+".rve.hdf5", "r") as fh5:
    
	gPhi_jj = fh5["gPhi_jj"][()]   
	gPhi_ij = fh5["gPhi_ij"][()]   
	gPhi_ii = fh5["gPhi_ii"][()]   
	phi_j   = fh5["phi_j"][()]    
	fh5.close()

## Diffusion data

In [7]:
T = 300 + 273          # Temperature [K] 
R = 8.31446261815324   # Universal gas constant [J/mol.K]

dt = 10              # Time increment [s]

# Lattice diffusivity mm²/s
D01 = 2.2e-4*1e6; DQ1 = 122500  
DL = D01*np.exp(-DQ1/(R*T)) 

# High solubility phase diffusivity
D02 = 1.5e-5*1e6; DQ2 = 142000
DT = D02*np.exp(-DQ2/(R*T))

print("DL: ", DL)
print("DT: ", DT)
print("DL/DT: ", DL/DT)

DL:  1.498129201501181e-09
DT:  1.7046448737059546e-12
DL/DT:  878.8512050866046


### Solubility parameters

In [8]:
# Interface enthalpies, all set to zero
zeta_ii = 0 
zeta_ij = 0
zeta_jj = 0

Hj = 26939.6934/2       # Partitioning enthalpy difference [J/mol]
S_over_R = 0.62715      # Partitioning entropy difference over R ΔS/R

zeta_j = Hj
print(f"Partitioning coefficient k({T} K): ", np.exp(zeta_j/(R*T) + S_over_R))

Partitioning coefficient k(573 K):  31.641283670894794


### Carbide trapping

In [9]:
f_j = 0.04      # vol fraction of carbide in martensite
C_stoic = 0.333 # Carbon stoichiometric concentration in Fe2C

Xmax = C_stoic*f_j # maximum carbon capacity in carbide

kf0 = 10.0  # Precipitation pre-exponent [s⁻1]
Qf = 2000   # Precipitation activation energy [J/mol]
kb0 = 0     # Dissolution pre-exponent
Qb = 0      # Dissolution activation energy
 
kf = kf0*np.exp(-Qf/(R*T))
print('kf: ', kf)
kb = kb0*np.exp(-Qb/(R*T))
print('kb: ', kb)
print('Xmax: ', Xmax)

kf:  6.571790335897728
kb:  0.0
Xmax:  0.01332


In [10]:
# Material `dict`

DiffMaterials = {
    mesh.getMaterialNames()[0] : {
		"D0x1" : D01,
		"D0y1" : D01,
		"DQx1" : DQ1,
		"DQy1" : DQ1,
		"D0x2" : D02,
		"D0y2" : D02,
		"DQx2" : DQ2,
		"DQy2" : DQ2,
		"zeta_j"   : zeta_j,
  		"S_over_R" : S_over_R,
		"zeta_jj"  : zeta_jj,
		"zeta_ij"  : zeta_ij,
		"zeta_ii"  : zeta_ii,
  		"Xmax" : Xmax,
  		"kf0"  : kf0,
		"Qf"   : Qf,
		"kb0"  : kb0,
		"Qb"   : Qb,
    }
}

### Apply boundary conditions

In [11]:
w_C = 0.4/100     # Carbon concentration in wt%

M_C = 12.011e-3   # [kg/mol]
M_Fe = 55.845e-3  # [kg/mol]
rho_Fe = 7874

print(rho_Fe/M_Fe)

# Convert to mole fraction 
x_C = (w_C / M_C) / ( (w_C / M_C) + ((1 - w_C) / M_Fe) )
print(x_C)

Con_b = x_C

print(Con_b)


140997.4035276211
0.018330366075578416
0.018330366075578416


In [12]:
# Dirichlet BCs list

exitNods = []
conBCs = []


# # Entry con
# conBCs.extend(AssignDirichletBC(mesh.getNodesByGroup("left"), dof=0, value=0))

# # Exit con
# conBCs.extend(AssignDirichletBC(mesh.getNodesByGroup("right"), dof=0, value=0))

# conBCs = [[row[0], row[-1]] for row in conBCs]

### Initialize simulation object

In [13]:
# Create the config object
diffPhysConfig = PhysicsConfig(
    SimulName = SimulName,
    PhysicsType = "Transport",
    PhysicsCategory = SimulType,
    nSteps    = nSteps,
    presBCs=conBCs,
    exitNodes = exitNods,
    dt = dt,
    Temperature = T,
)

In [14]:
DiffRA = PP(diffPhysConfig, meshConfig, DiffMaterials)


PHIMATS: 300_eq_f4_k1_dis | Transport (2PhaseTrapping)
  !! WARNING: INSULATED (No concentration BCs) !!
  Nodes/Elems : 288106 / 574912
  Total DOFs  : 288106
  BCs Count   : 0



In [15]:
DiffRA.WriteInputFile()

  Input file initialized: 300_eq_f4_k1_dis.diff.in.hdf5


In [16]:
DiffRA.WriteOutputFile(OVERWRITE=True, AVCON=True, TRAPPING=True)

  Output file initialized: 300_eq_f4_k1_dis.diff.out.hdf5


### Generate xdmf visualization file

In [17]:
WriteXDMF(SimulName, # Base simulation name
          "RVE",
          elementName, # Element name
          nSteps+1, # Number of steps 
          components=["diff"],
          TRAPPING=True,
		  nDim=2)